# Introdução à Inteligência Artificial 2026/2027

## Projeto 1: Masmorra da Perdição

### Entrega: 6 de Outubro às 23:59

## Introdução

Considere o jogo clássico *Rogue*, que decorre na Masmorra da Perdição, uma masmorra bidimensional vista de cima. A masmorra é delimitada por paredes rígidas de pedra e pode conter obstáculos internos, monstros que guardam os corredores e o mítico **Amuleto de Yendor**. O herói explora a masmorra movimentando-se livremente nas quatro direções cardinais.

Contudo, a masmorra é perigosa: para além de contornar paredes, o herói pode ter de derrotar monstros para avançar. Cada monstro tem um determinado nível de pontos de vida (HP). Quando o herói tenta mover-se para a célula de um monstro, realiza um ataque que reduz o HP desse monstro em 1 unidade, mantendo-se na sua posição atual. O monstro só é eliminado (libertando a célula) quando o seu HP chega a 0. O herói dispõe de uma reserva finita de energia/vida para a missão. O objetivo é fazer o herói chegar à célula do Amuleto de Yendor com vida suficiente e no menor número de ações. Neste projeto queremos fazer **apenas a formulação** do problema.

## Representação da Masmorra da Perdição

Considere que cada célula da masmorra é dada por um par de coordenadas $(x,y)$, onde **a célula no topo à esquerda é a célula $(0,0)$** e **a célula em baixo à direita é $(M-1,N-1)$**, com $M$ o número de colunas e $N$ o número de linhas da masmorra.

Na representação da Masmorra considera-se que:
* as paredes são representadas pelo símbolo "="
* o Amuleto de Yendor é representado pelo símbolo "Y"
* as casas livres são representadas por "."
* o herói é representado por "@"
* os monstros são representados por dígitos numéricos de "1" a "9", que indicam o seu HP atual (ex.: "2" representa um monstro que precisa de 2 ataques para ser derrotado)

Abaixo apresenta-se uma visualização, utilizando os símbolos descritos acima, de uma masmorra de dimensão 19x8, onde o herói está na célula (12,4), um monstro com 2 de HP está na célula (14,4) e o amuleto está na célula (16,6).

```
= = = = = = = = = = = = = = = = = = =
= . . . . . . . . . . . . . . . . . =
= . . . . . . . . . . . . . . . . . =
= . . . . . . . . . . . . = . . . . =
= . . . . . . . . . . . @ . 2 . . . =
= . . . . . . . = . . . . = = . . . =
= . . . . . . . = . . . . = = . Y . =
= = = = = = = = = = = = = = = = = = =
```

## Objetivo do Projeto

Formular este problema como um problema de procura no Espaço de Estados, usando a implementação disponibilizada pelo módulo `searchPlus.py` que é fornecido juntamente com este enunciado, assim como `utils.py`. O estado deve ser mínimo, contendo apenas os elementos que mudam com as ações.

Assim, os alunos terão de completar a classe `MasmorraRogue`:

```python
from searchPlus import *

# predefinição de uma masmorra inicial simples:
line1 = "= = = = = = =\n"
line2 = "= Y . . . . =\n"
line3 = "= . . . = . =\n"
line4 = "= . . 1 = . =\n"
line5 = "= = = . = . =\n"
line6 = "= @ . . . . =\n"
line7 = "= = = = = = =\n"
grid = line1 + line2 + line3 + line4 + line5 + line6 + line7

class MasmorraRogue(Problem):

    def __init__(self, mundo=grid, vida_inicial=10):
        pass

    def actions (self, state):
        pass

    def result (self, state, action):
        pass

    def goal_test (self, state):
        pass

    def display (self, state):
        pass

    def executa(self, state, actions_list, verbose=False):
        """Executa uma sequência de ações a partir do estado, devolvendo o triplo formado pelo estado 
        final, o custo acumulado e o booleano que indica se o objectivo foi ou não atingido. Se o
        objectivo for atingido antes da sequência ser toda executada, devolve-se o estado e o custo
        corrente. Há o modo verboso e o não verboso, este último selecionado por defeito."""
        cost = 0
        for a in actions_list:
            seg = self.result(state,a)
            cost = self.path_cost(cost,state,a,seg)
            state = seg
            obj = self.goal_test(state)
            if verbose:
                print('Ação:', a)
                print(self.display(state),end='')
                print('Custo Total:',cost)
                print('Atingido o objetivo?', obj)
                print()
            if obj:
                break
        return (state, cost, obj)
```

#### O construtor de `MasmorraRogue`

O construtor da classe a desenvolver recebe como input a informação em texto (como exemplificado acima) referente à masmorra inicial e o nível de vida inicial do herói. As dimensões da masmorra, a localização das paredes, do amuleto, dos monstros (e respetivos HPs) e do herói são dados de forma implícita no texto. Será necessário extrair essas informações explicitamente.

#### O método `actions`

As ações correspondem às direções em que o herói se pode tentar movimentar, e são quatro: Baixo, Direita, Cima e Esquerda, identificadas pelos símbolos "B", "D", "C" e "E", respetivamente. Cada ação afeta a célula ortogonalmente adjacente. Nem todas as ações são possíveis em todas as situações:

* O herói não pode mover-se para células ocupadas por paredes ("=");
* O herói pode mover-se para células livres (".") ou para a célula do amuleto ("Y");
* O herói pode tentar mover-se para uma célula ocupada por um monstro (dígito de 1 a 9); esta ação representa um **ataque** ao monstro tendo como consequência redução dos HPs do herói;
* O herói deve ter pelo menos 1 ponto de vida restante para poder realizar qualquer ação.

A lista de ações possíveis devolvidas por este método deverá estar **ordenada pela ordem definida das ações**.

##### Exemplos

**1)** Para a masmorra simples predefinida acima (com o herói na célula (1,5)), as ações possíveis no estado inicial são movimentar-se para a direita ou para cima:
```python
p = MasmorraRogue()
print(p.actions(p.initial))
```
gerando o output
```python
['D', 'C']
```

**2)** Se a masmorra inicial tiver um monstro adjacente ao herói:
```python
line1 = "= = = = =\n"
line2 = "= . Y . =\n"
line3 = "= . . . =\n"
line4 = "= @ 3 . =\n"
line5 = "= = = = =\n"
grid2 = line1 + line2 + line3 + line4 + line5
```
a lista de ações do estado inicial é obtida por
```python
p = MasmorraRogue(grid2)
print(p.actions(p.initial))
```
gerando o output (onde 'D' representa atacar o monstro e 'C' mover para cima)
```python
['D', 'C']
```

**3)** Caso a vida do herói chegue a 0, não existem ações possíveis:
```python
p = MasmorraRogue(grid2, vida_inicial=0)
print(p.actions(p.initial))
```
gerando o output
```python
[]
```

#### O método `goal_test`

Chega-se ao objetivo quando a posição do herói coincide com a célula onde está o Amuleto de Yendor.

#### O método `result`

Repare-se que, se um estado $s$ for argumento do método `result`, ele deve permanecer exatamente igual, não sendo modificado pelo método. Deve ser gerado um novo estado $s'$ sem alterar $s$.

Ao aplicar uma ação:
- **Movimento para célula livre/amuleto:** O herói muda para a nova célula e perde um 1 HP. A célula anterior fica livre (".").
- **Ação contra monstro (Ataque):** O herói **não se desloca**. O monstro na célula visada tem o seu HP reduzido em 1 unidade. Se o HP do monstro chegar a 0, ele é derrotado e a célula passa a estar livre ("."). O herói perde 1 ponto de vida por cada troca de golpes.

#### Igualdade entre estados

Dois estados com os mesmos valores nos seus atributos (posição do herói, vida restante, e posições/HP dos monstros ativos) são considerados iguais mesmo que sejam objetos distintos!

#### Sem programação defensiva

Os alunos apenas precisam de ter em consideração situações iniciais de masmorras que sejam válidas: masmorras quadradas ou retangulares delimitadas por paredes, sem símbolos desconhecidos, contendo exatamente um herói e um amuleto. Assuma que o input do construtor da classe `MasmorraRogue` é sempre válido.

#### O método `display`

A função `display` pega no estado e faz a visualização da masmorra em modo texto, respeitando o formato usado no input do construtor. Um exemplo para o mundo predefinido inicial:
```python
p = MasmorraRogue()
print(p.display(p.initial))
```
gera o seguinte output:
```
= = = = = = =
= Y . . . . =
= . . . = . =
= . . 1 = . =
= = = . = . =
= @ . . . . =
= = = = = = =
```

Quando o herói atinge a casa do amuleto, a célula deve mostrar o herói ("@") e não o amuleto.

#### O método `executa`

A função `executa` permite executar uma sequência de ações a partir de um determinado estado, devolvendo: o estado que resulta da aplicação da sequência de ações; o custo total dessas ações (cada ataque/movimento tem custo 1); a indicação se o objetivo foi ou não satisfeito. Se o objetivo for atingido antes de aplicar todas as ações ou se o herói fica sem pontos de vida, o processo é interrompido.

O modo verboso apresenta o estado em modo texto após cada ação, o custo acumulado e o resultado do teste do objetivo.

Exemplo de teste:
```python
p = MasmorraRogue()
seq = ['D','D','C','C','C','C','C','E','E']
p.goal_test(p.executa(p.initial, seq)[0])
```
gera o output:
```python
True
```

## Submissão

Cada aluno deve completar a implementação da classe `MasmorraRogue` e testá-la no link do quizz **Projeto 1** na página da disciplina. Este projeto deve ser realizado **individualmente**.
O aluno pode fazer **Check** sempre que desejar sem penalização. Atenção que após carregar em **Finish attempt** o processo termina e não será possivel fazer alterações/correções.

O quizz é constituído pela implementação da classe `MasmorraRogue` e será avaliada através de um conjunto de testes automáticos visíveis e invisíveis (valendo um total de 20 valores).

### **Bom trabalho!**

